# Spam Detection — Error Type Breakdown

Which spam manipulation techniques are hardest for models to detect?

We load the archived GET pipeline runs and evaluate per error type — no new generation needed.
For each error type the ground-truth label is:
- **SPAM** for all manipulation types (urgency_phrase, phishing_link, etc.)
- **HAM** for paraphrase (legitimate rewrite)

## Setup

In [ ]:
import sys
import os
import json
import glob
from collections import defaultdict

project_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

## Load Models

In [ ]:
from framework.models.spam.roberta import RobertaSpamModel

MSHENODA_CONFIG = {
    "name": "mshenoda/roberta-spam",
    "type": "roberta",
    "max_length": 128,
    "label_map": {"LABEL_0": "HAM", "LABEL_1": "SPAM"}
}

MARIAGRANDURY_CONFIG = {
    "name": "mariagrandury/roberta-base-finetuned-sms-spam-detection",
    "type": "roberta",
    "max_length": 128,
    "label_map": {"LABEL_0": "HAM", "LABEL_1": "SPAM"}
}

print("Loading models...")
mshenoda      = RobertaSpamModel(MSHENODA_CONFIG)
mariagrandury = RobertaSpamModel(MARIAGRANDURY_CONFIG)
print("All models loaded.")

## Load Archived Generated Data

We use the latest pipeline session. Each run is a separate file — we pool all runs together.

In [ ]:
generated_dir = os.path.join(project_root, "framework", "data", "generated", "spam")

# Pick the most recent session (latest date prefix)
all_files = sorted(glob.glob(os.path.join(generated_dir, "*.json")))
latest_session = sorted(set(os.path.basename(f)[:15] for f in all_files))[-1]
session_files  = [f for f in all_files if os.path.basename(f).startswith(latest_session)]

print(f"Session: {latest_session}")
print(f"Files:   {[os.path.basename(f) for f in session_files]}")

# Pool all runs
samples = []
for path in session_files:
    with open(path, encoding="utf-8") as f:
        samples.extend(json.load(f))

print(f"Total samples: {len(samples)}")
print(f"Error types:   {sorted(set(s['error_type'] for s in samples))}")

## Run Models & Group by Error Type

In [ ]:
def get_label(error_type: str) -> str:
    """Ground-truth label mirrors SpamTask.get_label()."""
    return "HAM" if error_type == "paraphrase" else "SPAM"

def evaluate_by_error_type(model, samples):
    texts       = [s["corrupted"] for s in samples]
    predictions = model.predict(texts)

    # Group correct/total counts per error type
    counts = defaultdict(lambda: {"correct": 0, "total": 0})
    for sample, pred in zip(samples, predictions):
        et    = sample["error_type"]
        label = get_label(et)
        counts[et]["total"]   += 1
        counts[et]["correct"] += int(pred == label)

    return {et: c["correct"] / c["total"] for et, c in counts.items()}

print("Evaluating mshenoda...")
results_mshenoda      = evaluate_by_error_type(mshenoda, samples)

print("Evaluating mariagrandury...")
results_mariagrandury = evaluate_by_error_type(mariagrandury, samples)

# Show table
import pandas as pd
error_types = sorted(results_mshenoda.keys())
df = pd.DataFrame({
    "mshenoda":      [results_mshenoda[et]      for et in error_types],
    "mariagrandury": [results_mariagrandury[et] for et in error_types],
}, index=error_types)
print(df.round(3).to_string())

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sort error types by mshenoda accuracy (ascending) so hardest types are on the left
error_types_sorted = sorted(error_types, key=lambda et: results_mshenoda[et])

x     = np.arange(len(error_types_sorted))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(x - width/2, [results_mshenoda[et]      for et in error_types_sorted], width, label="mshenoda",      color="#3498db")
ax.bar(x + width/2, [results_mariagrandury[et] for et in error_types_sorted], width, label="mariagrandury", color="#e67e22")

ax.axhline(y=0.5, color="grey", linestyle="--", linewidth=0.8, label="random baseline")

ax.set_ylabel("Accuracy")
ax.set_title("Detection Accuracy by Spam Error Type")
ax.set_xticks(x)
ax.set_xticklabels([et.replace("_", "\n") for et in error_types_sorted], fontsize=9)
ax.set_ylim(0, 1.15)
ax.legend()
ax.yaxis.grid(True, linestyle="--", alpha=0.7)

fig.tight_layout()
plt.savefig("error_type_breakdown.png", dpi=150)
plt.show()
print("Plot saved to error_type_breakdown.png")

## Interpretation

Error types sorted left to right from **hardest** to **easiest** to detect (by mshenoda accuracy).

- Types near the **left** (low accuracy) reveal blind spots — the model fails to flag these manipulation techniques
- **paraphrase** should score high since the correct label is HAM and models should classify legitimate text as HAM
- Large gaps between the two models on the same error type suggest one model was trained on more diverse spam examples